# 02. Functions and Scope

## What you'll learn

- How to define functions with parameters, default values, and return statements
- How to write clear **Google-style docstrings**
- How Python's **scope rules** work (local vs global, the LEGB rule)
- How to treat **functions as first-class values** — assign them, pass them, store them in dicts
- How to use **lambda expressions** for concise one-liners
- How to use **`*args` and `**kwargs`** for flexible function signatures

## Prerequisites

- [01. Python Fundamentals](./01_python_fundamentals.ipynb) — variables, basic types, `if`/`for`/`while`, basic `print()`

## Why this matters for agents

Every AI agent is, at its core, a loop that **calls functions**. Tools are functions. Prompt builders are functions. Parsers are functions. The dispatch logic that routes an LLM's chosen action to the right tool? That's a dictionary of functions. If you want to build agents from scratch, you need rock-solid intuition for how Python functions work — how they receive data, return results, and how you can pass them around like any other value.

> **Further reading:**
> - [Python docs — Defining Functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions) — the official tutorial, concise and authoritative
> - [Real Python — Defining Your Own Python Function](https://realpython.com/defining-your-own-python-function/) — thorough walkthrough with examples

---
## 1. Defining Functions: `def`, Parameters, and `return`

A function packages a reusable piece of logic. You define it with `def`, give it a name, list its parameters, and use `return` to send a value back to the caller.

In agent systems, even the simplest operations — formatting a message, calling an API, parsing a response — live inside functions.

In [ ]:
# A simple function that formats a chat message dict
def format_message(role, content):
    """Return a message dict in the format LLM APIs expect."""
    return {"role": role, "content": content}


# Call it
msg = format_message("user", "What is the capital of France?")
print(msg)

### Default parameter values

You can give parameters default values so callers don't have to specify every argument every time. This is extremely common in agent code — think of optional `temperature`, `max_tokens`, or `model` parameters.

In [ ]:
def build_api_payload(messages, model="openai/gpt-3.5-turbo", temperature=0.7, max_tokens=256):
    """Build a payload dict for an OpenRouter-style API call."""
    return {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }


# Using defaults
payload_default = build_api_payload([format_message("user", "Hello!")])
print("With defaults:", payload_default)

# Overriding some defaults
payload_custom = build_api_payload(
    [format_message("user", "Hello!")],
    model="anthropic/claude-3-haiku",
    temperature=0.0,
)
print("Custom:      ", payload_custom)

### Returning multiple values

Python functions can return multiple values as a tuple. This is handy when a function needs to produce several related results — for example, both the parsed tool name *and* its arguments from an LLM response.

In [ ]:
def parse_tool_call(response_text):
    """Parse a simple 'tool_name: arg1, arg2' format.
    
    Returns the tool name and a list of arguments.
    """
    parts = response_text.split(":")
    tool_name = parts[0].strip()
    args = [a.strip() for a in parts[1].split(",")] if len(parts) > 1 else []
    return tool_name, args


name, arguments = parse_tool_call("calculator: 2, +, 3")
print(f"Tool: {name}")
print(f"Args: {arguments}")

---
## 2. Docstrings (Google Style)

A **docstring** is a string literal placed as the first statement in a function (or class, or module). It documents what the function does, what it takes, and what it returns. Good docstrings are essential when you're building a library of tools — both for human readers and, increasingly, for LLMs that need to understand what your tools do.

We'll use **Google style** — it's clean, readable, and widely used in the Python ecosystem.

In [ ]:
def search_web(query, num_results=5):
    """Search the web and return a list of result snippets.

    This is a simulated tool that an agent might use to retrieve
    information from the internet.

    Args:
        query: The search query string.
        num_results: Maximum number of results to return. Defaults to 5.

    Returns:
        A list of dicts, each with 'title' and 'snippet' keys.
    """
    # Simulated results for demonstration
    return [
        {"title": f"Result {i} for '{query}'", "snippet": f"This is snippet {i}."}
        for i in range(1, num_results + 1)
    ]


# Python stores the docstring in the __doc__ attribute
print(search_web.__doc__)

In [ ]:
# You can also use help() to view it nicely formatted
help(search_web)

Why does this matter for agents? When you register tools for an LLM to use, the agent framework often extracts the **docstring** automatically to tell the LLM what the tool does. Clear docstrings lead to better tool use.

---
## 3. Scope: Local vs Global (the LEGB Rule)

When you use a variable name inside a function, Python looks it up in a specific order:

1. **L**ocal — inside the current function
2. **E**nclosing — inside any enclosing (outer) functions
3. **G**lobal — at the module/file level
4. **B**uilt-in — Python's built-in names (`print`, `len`, `range`, etc.)

This is called the **LEGB rule**. Understanding it prevents subtle bugs, especially when you have configuration values (like model names or API keys) that live at the module level.

In [ ]:
# Global variable — module-level configuration
DEFAULT_MODEL = "openai/gpt-3.5-turbo"


def get_model_name():
    # No local variable named DEFAULT_MODEL, so Python finds the global one
    return DEFAULT_MODEL


def get_model_name_local():
    # This creates a LOCAL variable that shadows the global
    DEFAULT_MODEL = "anthropic/claude-3-haiku"
    return DEFAULT_MODEL


print("Global lookup:", get_model_name())
print("Local shadow: ", get_model_name_local())
print("Global still: ", DEFAULT_MODEL)  # unchanged!

### Enclosing scope (closures)

When a function is defined *inside* another function, it can access variables from the enclosing function's scope. This pattern — called a **closure** — is useful for creating specialized functions on the fly.

In [ ]:
def make_system_prompt_builder(persona):
    """Return a function that builds a system message with a fixed persona."""
    def build_prompt(task):
        # 'persona' comes from the enclosing scope
        return format_message(
            "system",
            f"You are {persona}. Your current task: {task}"
        )
    return build_prompt


# Create specialized builders
code_assistant = make_system_prompt_builder("a helpful coding assistant")
math_tutor = make_system_prompt_builder("a patient math tutor")

print(code_assistant("Write a fizzbuzz function"))
print(math_tutor("Explain what a derivative is"))

### A word of caution: mutable default arguments

One famous Python gotcha — default argument values are evaluated **once** when the function is defined, not each time it's called. If the default is a mutable object (like a list or dict), it's shared across all calls.

In [ ]:
# BAD: mutable default argument
def add_message_bad(messages=[], role="user", content="hello"):
    messages.append({"role": role, "content": content})
    return messages


result1 = add_message_bad()  # starts with []
result2 = add_message_bad()  # SURPRISE: starts with the previous call's list!
print("Oops, shared state:", result2)  # has TWO messages


# GOOD: use None as default, create a new list inside
def add_message_good(role="user", content="hello", messages=None):
    if messages is None:
        messages = []
    messages.append({"role": role, "content": content})
    return messages


result3 = add_message_good()
result4 = add_message_good()
print("Correct, fresh list:", result4)  # just ONE message

---
## 4. Functions as First-Class Values

In Python, functions are **objects** — you can assign them to variables, store them in data structures, and pass them as arguments to other functions. This is the foundation for building **tool dispatch tables** in agent systems.

When an LLM says "use the calculator tool", your agent needs to look up the right function and call it. That lookup is typically a dictionary mapping tool names to functions.

In [ ]:
# Functions are values — you can assign them to variables
def greet(name):
    return f"Hello, {name}!"


# Assign the function to another name (no parentheses = no call)
say_hello = greet
print(say_hello("Agent"))  # same function, different name
print(type(say_hello))     # it's a function object

### Passing functions as arguments (higher-order functions)

A function that takes another function as an argument is called a **higher-order function**. This pattern appears constantly — think of `map()`, `filter()`, `sorted()` with a `key`, or an agent loop that accepts a "strategy" function.

In [ ]:
def apply_to_messages(messages, transform_fn):
    """Apply a transform function to each message's content."""
    return [
        {"role": m["role"], "content": transform_fn(m["content"])}
        for m in messages
    ]


def uppercase(text):
    return text.upper()


def add_prefix(text):
    return f"[PROCESSED] {text}"


conversation = [
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a programming language."},
]

print("Uppercased:", apply_to_messages(conversation, uppercase))
print("Prefixed:  ", apply_to_messages(conversation, add_prefix))

### Dispatch dictionaries

This is the big one for agents. A **dispatch dictionary** maps string names to functions. When the LLM picks a tool by name, you look it up in the dict and call it.

In [ ]:
# Define some simple tool functions
def calculator(expression):
    """Evaluate a simple math expression."""
    # In production you'd use a safe parser — eval is just for demo
    allowed_chars = set("0123456789+-*/.() ")
    if all(c in allowed_chars for c in expression):
        return str(eval(expression))
    return "Error: invalid expression"


def get_weather(city):
    """Get the current weather for a city (simulated)."""
    fake_data = {"paris": "22C, sunny", "london": "15C, rainy", "tokyo": "28C, humid"}
    return fake_data.get(city.lower(), f"No data for {city}")


def word_count(text):
    """Count the number of words in the given text."""
    return str(len(text.split()))


# The dispatch dict: maps tool names (strings) to functions
tools = {
    "calculator": calculator,
    "get_weather": get_weather,
    "word_count": word_count,
}

# Simulating what an agent loop does:
tool_name = "calculator"   # <-- LLM chose this
tool_input = "(2 + 3) * 7" # <-- LLM provided this

result = tools[tool_name](tool_input)
print(f"Tool '{tool_name}' returned: {result}")

---
## 5. Lambda Expressions

A **lambda** is a small anonymous function defined in a single expression. It's useful when you need a quick throwaway function — for example, as a `key` for sorting, or a simple transform.

```python
lambda parameters: expression
```

Lambdas can only contain a single expression (no statements, no assignments). If your logic needs more than one line, use a regular `def`.

In [ ]:
# Lambda basics
double = lambda x: x * 2
print(double(5))  # 10

# Commonly used inline — sorting messages by content length
messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello! How can I help you today?"},
    {"role": "user", "content": "Tell me about Python."},
]

sorted_msgs = sorted(messages, key=lambda m: len(m["content"]))
for m in sorted_msgs:
    print(f"  [{len(m['content']):2d} chars] {m['role']}: {m['content']}")

In [ ]:
# Lambdas with map() and filter()
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Double all numbers
doubled = list(map(lambda x: x * 2, numbers))
print("Doubled:", doubled)

# Keep only even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print("Evens:  ", evens)

# Filter messages to only user messages
user_msgs = list(filter(lambda m: m["role"] == "user", messages))
print("User messages:", user_msgs)

---
## 6. `*args` and `**kwargs`

Sometimes you don't know in advance how many arguments a function will receive. Python provides two special syntaxes:

- **`*args`** — collects extra *positional* arguments into a **tuple**
- **`**kwargs`** — collects extra *keyword* arguments into a **dict**

These are essential for writing flexible wrappers and dispatchers. Agent tools have varying signatures — a dispatcher needs to forward arbitrary arguments to whichever tool the LLM chose.

In [ ]:
# *args: variable positional arguments
def build_conversation(*messages):
    """Build a conversation list from any number of (role, content) tuples."""
    return [{"role": role, "content": content} for role, content in messages]


conversation = build_conversation(
    ("system", "You are a helpful assistant."),
    ("user", "What is 2+2?"),
    ("assistant", "2+2 equals 4."),
    ("user", "Thanks!"),
)
for msg in conversation:
    print(f"  {msg['role']:>10}: {msg['content']}")

In [ ]:
# **kwargs: variable keyword arguments
def create_tool_definition(name, description, **parameters):
    """Create a tool definition dict with arbitrary parameters.

    Args:
        name: The tool's name.
        description: What the tool does.
        **parameters: Each keyword becomes a parameter definition.

    Returns:
        A tool definition dict.
    """
    return {
        "name": name,
        "description": description,
        "parameters": parameters,
    }


weather_tool = create_tool_definition(
    "get_weather",
    "Get the current weather for a city.",
    city="string: the city name",
    units="string: 'celsius' or 'fahrenheit', default 'celsius'",
)
print(weather_tool)

In [ ]:
# Combining *args and **kwargs: the universal forwarder
def log_and_call(func, *args, **kwargs):
    """Call a function with any arguments, logging the call details."""
    print(f"Calling {func.__name__}(args={args}, kwargs={kwargs})")
    result = func(*args, **kwargs)
    print(f"  -> returned: {result}")
    return result


# Works with any function and any arguments
log_and_call(calculator, "3 + 4 * 2")
log_and_call(get_weather, city="Paris")
log_and_call(word_count, "The quick brown fox jumps over the lazy dog")

### Unpacking with `*` and `**`

The `*` and `**` operators also work in the *other* direction — you can unpack a list/tuple into positional args, or a dict into keyword args. This is how dispatchers forward stored arguments to tool functions.

In [ ]:
# Unpacking a dict as keyword arguments
tool_call = {
    "name": "get_weather",
    "arguments": {"city": "Tokyo"},
}

# Instead of: get_weather(city="Tokyo")
# We can do:
result = tools[tool_call["name"]](**tool_call["arguments"])
print(f"Result: {result}")

---
## Putting It Together: A Tool Registry and Dispatcher

Let's combine everything we've learned into a mini tool-dispatch system — the kind of thing that sits at the heart of every agent.

We'll build:
1. A **tool registry** — a dict mapping names to functions, with metadata
2. A **`register_tool`** function — adds a tool to the registry using its name and docstring
3. A **`dispatch`** function — looks up a tool by name and calls it with `**kwargs`
4. **Error handling** — graceful messages for unknown tools or bad arguments

In [ ]:
# The tool registry: maps tool names to {"function": ..., "description": ...}
tool_registry = {}


def register_tool(func):
    """Register a function as a tool, using its name and docstring.

    Args:
        func: The function to register.

    Returns:
        The function, unchanged (so this can be used as a decorator later).
    """
    tool_registry[func.__name__] = {
        "function": func,
        "description": (func.__doc__ or "No description available.").strip(),
    }
    return func


def dispatch(tool_name, **kwargs):
    """Look up a tool by name and call it with the given keyword arguments.

    Args:
        tool_name: The name of the tool to call.
        **kwargs: Arguments to pass to the tool function.

    Returns:
        The tool's return value, or an error message string.
    """
    if tool_name not in tool_registry:
        available = ", ".join(tool_registry.keys())
        return f"Error: Unknown tool '{tool_name}'. Available tools: {available}"

    tool_func = tool_registry[tool_name]["function"]
    try:
        return tool_func(**kwargs)
    except TypeError as e:
        return f"Error calling '{tool_name}': {e}"


def list_tools():
    """Print all registered tools and their descriptions."""
    print("Available tools:")
    print("-" * 50)
    for name, info in tool_registry.items():
        print(f"  {name}: {info['description']}")
    print("-" * 50)

In [ ]:
# Register our tools
register_tool(calculator)
register_tool(get_weather)
register_tool(word_count)

# See what's registered
list_tools()

In [ ]:
# Simulate an agent processing a sequence of tool calls from an LLM
llm_tool_calls = [
    {"tool": "calculator", "args": {"expression": "(10 + 5) * 3"}},
    {"tool": "get_weather", "args": {"city": "London"}},
    {"tool": "word_count", "args": {"text": "The agent dispatched three tool calls"}},
    {"tool": "unknown_tool", "args": {}},  # This should produce an error
    {"tool": "calculator", "args": {"bad_param": "oops"}},  # Wrong arguments
]

print("Agent processing tool calls:")
print("=" * 60)
for i, call in enumerate(llm_tool_calls, 1):
    result = dispatch(call["tool"], **call["args"])
    status = "OK" if not str(result).startswith("Error") else "FAIL"
    print(f"  [{i}] {call['tool']:15s} -> [{status}] {result}")

This is essentially what happens inside every agent framework: the LLM produces a tool name and arguments, the agent looks up the function, calls it, and feeds the result back. You just built the core of that system with plain Python functions and a dictionary.

---
## Try It Yourself

Work through these exercises to solidify your understanding. Each one builds on concepts from this notebook.

### Exercise 1: Mean function

Write a function `mean(*numbers)` that takes any number of numeric arguments and returns their arithmetic mean. It should return `0.0` if no arguments are given.

```python
# Expected behavior:
mean(1, 2, 3)       # -> 2.0
mean(10, 20)         # -> 15.0
mean()               # -> 0.0
mean(7)              # -> 7.0
```

In [ ]:
# Exercise 1: your code here
def mean(*numbers):
    pass  # Replace with your implementation


# Uncomment to test:
# assert mean(1, 2, 3) == 2.0
# assert mean(10, 20) == 15.0
# assert mean() == 0.0
# assert mean(7) == 7.0
# print("All tests passed!")

### Exercise 2: Call-N-times higher-order function

Write a function `call_n_times(func, n, *args, **kwargs)` that calls `func` exactly `n` times with the given arguments, collecting and returning all results in a list.

This pattern is useful for retrying API calls or running a tool multiple times with the same input.

```python
# Expected behavior:
call_n_times(calculator, 3, expression="1 + 1")
# -> ["2", "2", "2"]
```

In [ ]:
# Exercise 2: your code here
def call_n_times(func, n, *args, **kwargs):
    pass  # Replace with your implementation


# Uncomment to test:
# results = call_n_times(calculator, 3, expression="1 + 1")
# assert results == ["2", "2", "2"]
# results = call_n_times(get_weather, 2, city="Paris")
# assert results == ["22C, sunny", "22C, sunny"]
# print("All tests passed!")

### Exercise 3: Extended tool registry

Build a fresh tool registry with **3 new tools** of your own design (ideas: `reverse_text`, `count_vowels`, `title_case`, `char_frequency`, etc.). Then write a `run_tool_sequence` function that:

1. Takes a list of tool-call dicts (like `llm_tool_calls` above)
2. Dispatches each one through the registry
3. Collects results and returns a list of `{"tool": ..., "status": "ok"/"error", "result": ...}` dicts

Include at least one test call to an unknown tool to verify error handling.

In [ ]:
# Exercise 3: your code here

# Step 1: Create a new registry and define 3 tool functions
my_registry = {}

# Step 2: Register your tools in my_registry

# Step 3: Write run_tool_sequence(calls, registry)

# Step 4: Test it with a list of calls (include one unknown tool)


### Exercise 4 (Stretch): Decorator-based registration

If you're feeling adventurous, notice that `register_tool` already returns the function unchanged — that means you can use it as a **decorator**. Try defining a new tool using `@register_tool` syntax:

```python
@register_tool
def my_new_tool(arg):
    """Description of my tool."""
    return f"processed: {arg}"
```

Verify it appears in the registry with `list_tools()`. We'll cover decorators in depth in [08. Decorators and Type Hints](./08_decorators_and_type_hints.ipynb).

In [ ]:
# Exercise 4 (Stretch): your code here


---

**Up next:** [03. Data Structures](./03_data_structures.ipynb) — lists, dicts, sets, and tuples in depth, with a focus on the data structures you'll use constantly in agent code (message lists, tool definitions, configuration dicts).